# 10 — Generative Model Evaluation

Evaluating generative models requires metrics that capture both quality and diversity.

## FID, IS, CLIP Score, Precision/Recall

**Fréchet Inception Distance (FID)**: compares the statistics of real and generated images in the feature space of an Inception network. Computes the Fréchet distance between two Gaussians fit to the feature vectors:
$$FID = ||\mu_r - \mu_g||^2 + \text{Tr}(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2})$$
Lower FID = more realistic; FID is sensitive to both quality and diversity (if the generator mode collapses, FID rises).

**Inception Score (IS)**: measures both quality (low entropy of p(y|x)) and diversity (high entropy of the marginal p(y)):
$$IS = \exp(\mathbb{E}_x[KL(p(y|x)||p(y))])$$
IS doesn't compare to real data — it only measures self-consistency of the generated distribution.

**CLIP Score**: for text-to-image, measures alignment between generated image and prompt using CLIP embeddings:
$$CLIP\text{-}S = \max(100 \cdot \cos(E_I(x), E_T(c)), 0)$$

**Precision and Recall** (Sajjadi et al., 2018; Kynkäänniemi et al., 2019):
- *Precision*: fraction of generated samples that fall in the real data manifold (quality)
- *Recall*: fraction of real data that can be recovered from the generated distribution (coverage/diversity)

In [ ]:
# Evaluation suite implementation
import numpy as np
import torch
from scipy.linalg import sqrtm

def compute_fid(real_features, fake_features):
    """Compute FID between real and fake feature distributions."""
    mu_r = real_features.mean(axis=0)
    mu_g = fake_features.mean(axis=0)
    sigma_r = np.cov(real_features, rowvar=False)
    sigma_g = np.cov(fake_features, rowvar=False)

    diff = mu_r - mu_g
    covmean, _ = sqrtm(sigma_r @ sigma_g, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma_r + sigma_g - 2 * covmean)
    return float(np.real(fid))


def compute_precision_recall(real_feat, fake_feat, k=5):
    """kNN-based precision and recall."""
    from sklearn.neighbors import NearestNeighbors

    # Precision: fraction of fake samples within real manifold
    nn_real = NearestNeighbors(n_neighbors=k).fit(real_feat)
    dist_fake_to_real, _ = nn_real.kneighbors(fake_feat)
    real_kth_dist = nn_real.kneighbors(real_feat)[0][:, -1]
    precision = (dist_fake_to_real[:, 0] <= real_kth_dist.mean()).mean()

    # Recall: fraction of real samples covered by fake manifold
    nn_fake = NearestNeighbors(n_neighbors=k).fit(fake_feat)
    dist_real_to_fake, _ = nn_fake.kneighbors(real_feat)
    fake_kth_dist = nn_fake.kneighbors(fake_feat)[0][:, -1]
    recall = (dist_real_to_fake[:, 0] <= fake_kth_dist.mean()).mean()

    return float(precision), float(recall)


# Simulate feature spaces for different generative models
np.random.seed(42)
n = 500
d = 32  # feature dimension

# Real features: two Gaussian clusters
real_feat = np.vstack([
    np.random.randn(n//2, d) + np.random.randn(d),
    np.random.randn(n//2, d) + np.random.randn(d) * 2,
])

# Good generator: covers both modes
good_fake = np.vstack([
    np.random.randn(n//2, d) + real_feat[:n//2].mean(0) + np.random.randn(d) * 0.1,
    np.random.randn(n//2, d) + real_feat[n//2:].mean(0) + np.random.randn(d) * 0.1,
])

# Mode collapsed generator: only covers one mode
collapsed_fake = np.random.randn(n, d) + real_feat[:n//2].mean(0) + np.random.randn(d) * 0.1

# Low quality generator: high variance noise
noisy_fake = np.random.randn(n, d) * 3

results = []
for name, fake in [('Good', good_fake), ('Mode Collapse', collapsed_fake), ('Low Quality', noisy_fake)]:
    fid = compute_fid(real_feat, fake)
    prec, rec = compute_precision_recall(real_feat, fake)
    results.append((name, fid, prec, rec))

print(f'{"Generator":<15} {"FID":>8} {"Precision":>10} {"Recall":>10}')
print('-'*45)
for name, fid, prec, rec in results:
    print(f'{name:<15} {fid:>8.1f} {prec:>10.3f} {rec:>10.3f}')

In [ ]:
# Evaluation visualisation
import matplotlib.pyplot as plt

names = [r[0] for r in results]
fids = [r[1] for r in results]
precs = [r[2] for r in results]
recs = [r[3] for r in results]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].bar(names, fids, color=['green', 'orange', 'red'])
axes[0].set_title('FID (lower = better)')
axes[0].set_ylabel('FID')

axes[1].bar(names, precs, color=['green', 'orange', 'red'])
axes[1].set_title('Precision (quality)')
axes[1].set_ylim(0, 1)

axes[2].bar(names, recs, color=['green', 'orange', 'red'])
axes[2].set_title('Recall (diversity/coverage)')
axes[2].set_ylim(0, 1)

plt.suptitle('Generative Model Evaluation Suite')
plt.tight_layout()
plt.savefig('/tmp/genmodel_eval.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nInterpretation:')
print('  Good: low FID, high precision AND recall')
print('  Mode collapse: low FID if one mode dominates; low recall (misses modes)')
print('  Low quality: high FID; low precision (generates out-of-distribution samples)')